# Notebook 04 — Objetivo Específico 2: Random Forest, Mapas a 10 m y Muestreo Estratificado (100% en SEPAL)

- **Instancia SEPAL recomendada:** **`m8d`** (8 CPUs, 32 GB RAM) o **`m16d`** (16 CPUs, 64 GB RAM). Todo el tuning RF y la predicción espacial completa sobre la costa corren en multi-núcleo dentro de tu máquina virtual de SEPAL (toma ~1–3 minutos y **cero** esperas de `Export.toDrive`).
- **Modelos comparados (ablación completa):**
  1. **`MS`**: Solo óptico Sentinel-2 (20 bandas).
  2. **`SAR`**: Solo radar Sentinel-1 ARD (4 bandas).
  3. **`MS_SAR_PCA`**: Combinado óptimo seleccionado por PCA en OE1.
  4. **`MS_SAR_FULL`**: Combinado completo (24 bandas S1+S2).
- **Salidas generadas localmente en SEPAL:**
  - Los 4 mapas GeoTIFF a 10 m UTM (`uint8` con paleta oficial del protocolo) en `outputs/mapas/`.
  - Mapa de 6 estratos de **Acuerdo/Desacuerdo en Manglar** + diseño muestral estratificado oficial tipo **Olofsson (2014) / AREA2** listo en CSV (`EPSG:4326`) para subir directo a **Collect Earth Online (CEO)**.

In [ ]:
# 1. Configuracion, datos 70/30 y definicion de los 4 modelos
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import geemap
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GridSearchCV, StratifiedKFold

from src.config_utils import ensure_output_dirs, get_class_maps, get_predictor_band_names, load_config
from src.local_aoi import ensure_local_aoi
from src.local_rf import (
    build_agreement_stratum_geotiff,
    generate_stratified_sample_ceo,
    predict_rf_to_geotiff,
)
from src.local_stack import add_local_raster_overlay, get_local_stack_paths
from src.metrics import compute_preliminary_metrics
from src.plotting import plot_confusion_matrix, plot_model_comparison

# >>> CAMBIAR AQUI ENTRE "PACIFICO" Y "CARIBE" <<<
REGION = "PACIFICO"

cfg = load_config()
dirs = ensure_output_dirs(cfg)
reg_l = REGION.lower()
ver = cfg["proyecto"]["version_pipeline"].lower()
SEED = int(cfg["proyecto"]["seed"])

CLASES_NOMBRES, CLASES_CORTOS, CLASES_COLORES = get_class_maps(cfg)
CLASS_IDS = sorted(CLASES_NOMBRES.keys())
BANDS = get_predictor_band_names(cfg)
paths_stk = get_local_stack_paths(REGION, cfg)

df_all = pd.read_csv(dirs["muestras"] / f"muestras_limpias_{reg_l}_70_30_con_bandas.csv")
sel_pca = json.loads((dirs["tablas"] / f"bandas_seleccionadas_pca_{reg_l}.json").read_text(encoding="utf-8"))

MODEL_SPECS = {
    "MS": BANDS["s2_todas"],
    "SAR": BANDS["sar_todas"],
    "MS_SAR_PCA": sel_pca["bandas_seleccionadas"],
    "MS_SAR_FULL": BANDS["todas"],
}

df_train = df_all[df_all["split"] == "train"].copy().reset_index(drop=True)
df_test = df_all[df_all["split"] == "test_preliminar"].copy().reset_index(drop=True)
print(f"Entrenamiento (70%): {len(df_train)} | Prueba preliminar (30%): {len(df_test)}")
for k, cols in MODEL_SPECS.items():
    print(f"  - {k:12s}: {len(cols)} bandas")

In [ ]:
# 2. Ajuste de hiperparametros Random Forest (GridSearchCV 5-fold en CPUs de SEPAL) + diagnostico 70/30
rf_cfg = cfg["clasificacion"]["rf"]
cv = StratifiedKFold(n_splits=int(rf_cfg["cv_folds"]), shuffle=True, random_state=SEED)

modelos_entrenados = {}
metricas_prelim = {}
resumen_prelim = []

y_train = df_train["clase_id"].values
y_test = df_test["clase_id"].values

for nombre_m, cols in MODEL_SPECS.items():
    grid = GridSearchCV(
        estimator=RandomForestClassifier(random_state=SEED, n_jobs=-1),
        param_grid=rf_cfg["param_grid"],
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        refit=True,
    )
    grid.fit(df_train[cols].values, y_train)
    best_rf = grid.best_estimator_
    y_pred = best_rf.predict(df_test[cols].values)

    met = compute_preliminary_metrics(y_test, y_pred, CLASS_IDS, CLASES_NOMBRES)
    m_row = met["per_class"].set_index("clase_id").loc[1]

    modelos_entrenados[nombre_m] = {"estimator": best_rf, "params": grid.best_params_, "cv_f1": grid.best_score_}
    metricas_prelim[nombre_m] = met
    resumen_prelim.append({
        "Modelo": nombre_m,
        "N_bandas": len(cols),
        "Mejores_params": str(grid.best_params_),
        "CV_F1_macro": grid.best_score_,
        "OA": met["OA"],
        "OA_CI_half": np.nan,
        "Kappa": met["Kappa"],
        "Kappa_CI_half": np.nan,
        "PU_Mangle": m_row["PU"],
        "PP_Mangle": m_row["PP"],
        "F1_Mangle": m_row["F1"],
        "F1_Mangle_CI_half": np.nan,
        "Macro_F1": met["Macro_F1"],
    })
    joblib.dump({"model": best_rf, "features": cols, "params": grid.best_params_}, dirs["modelos"] / f"rf_{nombre_m.lower()}_{reg_l}.joblib")

df_resumen_prelim = pd.DataFrame(resumen_prelim)
print(f"=== DIAGNOSTICO PRELIMINAR 70/30 (SOLO VERIFICACION DE MUESTRAS) — {REGION} ===")
display(df_resumen_prelim)
df_resumen_prelim.to_csv(dirs["tablas"] / f"OE2_diagnostico_preliminar_70_30_{reg_l}.csv", index=False)

In [ ]:
# 3. Figuras de diagnostico preliminar (70/30) e importancia por permutacion
plot_model_comparison(
    df_resumen_prelim,
    f"Desempeño Preliminar 70/30 (OA, Kappa y F1 Manglar) — {REGION}",
    save_path=dirs["figuras"] / f"OE2_prelim_comparacion_70_30_{reg_l}.png",
)
plt.show()

for nombre_m in ["MS", "SAR", "MS_SAR_PCA", "MS_SAR_FULL"]:
    plot_confusion_matrix(
        metricas_prelim[nombre_m]["confusion_matrix"],
        f"Matriz Preliminar (30%) — {nombre_m} ({REGION})",
        is_proportion=False,
        save_path=dirs["figuras"] / f"OE2_prelim_cm_{nombre_m.lower()}_{reg_l}.png",
    )
    plt.show()

perm = permutation_importance(
    modelos_entrenados["MS_SAR_FULL"]["estimator"],
    df_test[BANDS["todas"]].values,
    y_test,
    scoring="f1_macro",
    n_repeats=30,
    random_state=SEED,
    n_jobs=-1,
)
df_perm = pd.DataFrame(
    {"importancia_perm": perm.importances_mean, "std_perm": perm.importances_std},
    index=BANDS["todas"],
).sort_values("importancia_perm", ascending=False)
display(df_perm.head(12))
df_perm.to_csv(dirs["tablas"] / f"importancia_permutacion_rf_{reg_l}.csv")

In [ ]:
# 4. Clasificacion espacial completa a 10 m en CPU de SEPAL (4 mapas GeoTIFF + mapa de estratos)
# Corre 100% en tu maquina virtual (m8d/m16d) y calcula superficies exactas en hectareas.
block_sz = int(cfg.get("computo", {}).get("block_size", 1024))
map_paths = {m: dirs["mapas"] / f"mapa_{m.lower()}_{reg_l}_{ver}.tif" for m in MODEL_SPECS}
estratos_tif = dirs["mapas"] / f"mapa_estratos_area2_{reg_l}_{ver}.tif"

areas_por_modelo = {}
filas_areas = []
for nombre_m, cols in MODEL_SPECS.items():
    print(f"Clasificando mapa completo en SEPAL: {nombre_m} ({len(cols)} bandas)...")
    a_dict = predict_rf_to_geotiff(
        model=modelos_entrenados[nombre_m]["estimator"],
        stack_path=paths_stk["stack"],
        qa_path=paths_stk["qa"],
        feature_names=cols,
        out_tif_path=map_paths[nombre_m],
        class_colors=CLASES_COLORES,
        block_size=block_sz,
        mask_base_only=True,
    )
    areas_por_modelo[nombre_m] = a_dict
    fila = {"Modelo": nombre_m, "Total_AOI_ha": sum(a_dict.values())}
    for cid in CLASS_IDS:
        fila[f"{CLASES_CORTOS[cid]}_ha"] = a_dict.get(cid, 0.0)
    fila["Dif_vs_MARN2013_ha"] = a_dict.get(1, 0.0) - float(cfg["regiones"][REGION]["linea_base_marn_2013_ha"])
    filas_areas.append(fila)

areas_estratos_desac = build_agreement_stratum_geotiff(
    map_paths["MS"], map_paths["SAR"], map_paths["MS_SAR_PCA"], estratos_tif, block_size=block_sz
)

df_areas_oe2 = pd.DataFrame(filas_areas)
print(f"\n=== SUPERFICIE CARTOGRAFIADA (ha) — {REGION} (MARN 2013: {cfg['regiones'][REGION]['linea_base_marn_2013_ha']:,.2f} ha) ===")
display(df_areas_oe2)
print("Areas por estrato Acuerdo/Desacuerdo (ha):", {k: round(v, 2) for k, v in sorted(areas_estratos_desac.items())})

payload_areas = {
    "region": REGION,
    "areas_modelos_ha": areas_por_modelo,
    "areas_estratos_acuerdo_desacuerdo_ha": areas_estratos_desac,
    "map_paths": {k: str(v) for k, v in map_paths.items()},
    "estratos_tif": str(estratos_tif),
}
(dirs["tablas"] / f"areas_estratos_{reg_l}.json").write_text(json.dumps(payload_areas, indent=2), encoding="utf-8")
df_areas_oe2.to_csv(dirs["tablas"] / f"OE2_superficies_mapeadas_{reg_l}.csv", index=False)

In [ ]:
# 5. Diseno muestral estratificado tipo Olofsson / AREA2 directo en SEPAL (CSV para Collect Earth Online)
val_cfg = cfg["validacion_olofsson"]
ceo_design_csv = dirs["muestras"] / f"diseno_muestral_area2_para_CEO_{reg_l}.csv"
ceo_design_gpkg = dirs["muestras"] / f"diseno_muestral_area2_para_CEO_{reg_l}.gpkg"

gdf_diseno, df_cuotas = generate_stratified_sample_ceo(
    stratum_tif_path=estratos_tif,
    out_csv_path=ceo_design_csv,
    out_gpkg_path=ceo_design_gpkg,
    stratum_areas_ha=areas_estratos_desac,
    se_target_oa=float(val_cfg.get("error_estandar_objetivo_oa", 0.015)),
    min_per_stratum=int(val_cfg.get("n_min_por_estrato", 50)),
    seed=SEED,
)

print(f"=== DISENO MUESTRAL ESTRATIFICADO PARA COLLECT EARTH ONLINE ({REGION}) ===")
display(df_cuotas)
print(f"Total puntos disenados (EPSG:4326): {len(gdf_diseno)}")
print(f"[OK] Sube este CSV directo a Collect Earth Online (CEO): {ceo_design_csv}")

In [ ]:
# 6. Inspeccion visual interactiva de los mapas generados localmente en SEPAL
aoi_gdf = ensure_local_aoi(REGION, cfg)
b = aoi_gdf.total_bounds
m = geemap.Map(height="600px")
m.add_basemap("SATELLITE")
m.setCenter((b[0] + b[2]) / 2.0, (b[1] + b[3]) / 2.0, 11)

pal_5 = {cid: CLASES_COLORES[cid] for cid in CLASS_IDS}
pal_6 = {**pal_5, 6: "#984ea3"}

add_local_raster_overlay(m, paths_stk["stack"], ["B8_SECA", "B11_SECA", "B4_SECA"], "Stack Falsocolor (B8-B11-B4)", stretch=(0.0, 0.40), visible=False)
add_local_raster_overlay(m, map_paths["MS"], [1], "Mapa M1: Solo MS (S2)", categorical=True, palette=pal_5, visible=False)
add_local_raster_overlay(m, map_paths["SAR"], [1], "Mapa M2: Solo SAR (S1)", categorical=True, palette=pal_5, visible=False)
add_local_raster_overlay(m, map_paths["MS_SAR_PCA"], [1], "Mapa M3: Combinado MS+SAR (PCA)", categorical=True, palette=pal_5, visible=True)
add_local_raster_overlay(m, map_paths["MS_SAR_FULL"], [1], "Mapa M4: Combinado MS+SAR (24b)", categorical=True, palette=pal_5, visible=False)
add_local_raster_overlay(m, estratos_tif, [1], "Estratos CEO/AREA2 (Acuerdo/Desacuerdo)", categorical=True, palette=pal_6, visible=False)
m.add_legend(title="Coberturas", legend_dict={CLASES_NOMBRES[c]: CLASES_COLORES[c] for c in CLASS_IDS})
m